In [ ]:
# Configuração (células de código NÃO aparecem no PDF; as figuras são guardadas em figures/
# e referenciadas nas células de texto com ![Legenda](figures/x.pdf){#fig:id width=80%})
from pathlib import Path
import matplotlib.pyplot as plt
ROOT = Path.cwd()
FIG, DATA = ROOT / "figures", ROOT / "data"
FIG.mkdir(exist_ok=True)
plt.rcParams.update({"font.family": "sans-serif", "font.size": 9, "savefig.bbox": "tight", "figure.dpi": 150})

# Introdução

A função de uma proteína é determinada, em grande medida, pela sua estrutura tridimensional, que por sua vez está codificada na sequência de aminoácidos \cite{anfinsen1973}. Prever a estrutura a partir da sequência — o chamado *problema do enovelamento proteico* — foi durante cerca de 50 anos um dos grandes desafios em aberto da biologia. Na 14.ª edição do CASP (*Critical Assessment of protein Structure Prediction*), em 2020, o AlphaFold 2, desenvolvido pela Google DeepMind, atingiu pela primeira vez uma exatidão comparável à de métodos experimentais na maioria dos alvos \cite{jumper2021,kryshtafovych2021casp14}. Este avanço valeu a Demis Hassabis e John Jumper o Prémio Nobel da Química de 2024.

\afazer{Fechar a introdução: objetivo do trabalho (analisar AF2 e AF3 segundo os 4 eixos do enunciado + caso de estudo próprio).}

# Enquadramento Biológico e Contexto Ómico

## Ómica(s) abrangida(s)
\afazer{Biologia estrutural / proteómica estrutural; ligação a genómica e transcriptómica (sequências), interactómica (Multimer, AF3).}

## Dados de entrada e tecnologias associadas
\afazer{FASTA (aa); AF3: JSON com proteínas, DNA/RNA, ligandos (SMILES/CCD), iões, PTMs. Bases de dados (UniRef, MGnify, BFD, PDB). Dados de treino do PDB: cristalografia de raios-X, crio-EM, RMN. Origem das sequências: Illumina/PacBio/ONT, proteómica LC-MS/MS.}

## Pergunta biológica
\afazer{Qual a estrutura 3D da proteína? Como interage com outras moléculas? Aplicações: desenho de fármacos, interpretação de variantes, anotação funcional.}

# Arquitetura e Implementação Computacional

## Linguagens e bibliotecas

Ambas as versões estão escritas em **Python** e assentam na biblioteca de diferenciação automática **JAX**, com a camada de redes neuronais **Haiku** (ambas da Google). No AlphaFold 2 (AF2), a construção dos inputs recorre a ferramentas externas consagradas: **HMMER** (*jackhmmer*) e **HH-suite** (*HHblits*, *HHsearch*) para as pesquisas de homólogos e de *templates*, **Kalign** para alinhamentos e **OpenMM** para a relaxação final com o campo de forças Amber \cite{jumper2021,af2github}. O AlphaFold 3 (AF3) acrescenta componentes em **C++** (expostos ao Python via *pybind11*) para o processamento de dados e **RDKit** para a química de ligandos \cite{af3github}.

## Formas de acesso e instalação

Há quatro formas de usar o AlphaFold, que diferem bastante nos requisitos de hardware e no grau de controlo sobre a previsão (Tabela \ref{tab:acesso}).

\begingroup\small

| Via | Versão | Requisitos | Interface | Principais limitações |
|:-------------|:--------|:--------------------|:--------|:------------------------------|
| AlphaFold Server | AF3 | Browser e conta Google | Web | Quota diária; uso não comercial; ligandos e modificações restritos a uma lista |
| ColabFold | AF2 (+ Multimer) | Browser (GPU gratuita do Colab) ou instalação local | Notebook / CLI | Memória da GPU gratuita limita o tamanho; sessões com tempo limitado |
| AlphaFold DB | AF2 (pré-calculado) | Nenhum | Web / API REST | Só cadeias proteicas naturais; sem complexos, ligandos ou mutantes |
| Instalação local | AF2 e AF3 | Linux, GPU NVIDIA, 0,6--2,6 TB de bases de dados | CLI (Docker) | Hardware exigente; pesos do AF3 só mediante autorização |

: Formas de acesso ao AlphaFold. {#tab:acesso}

\endgroup

**AlphaFold Server.** Lançado em 2024 com o AF3 \cite{abramson2024af3}, é a forma mais simples de aceder à versão mais recente: no browser (\url{https://alphafoldserver.com}) introduzem-se as cadeias de proteína, DNA ou RNA, escolhem-se ligandos, iões e modificações pós-traducionais de uma lista predefinida, e em poucos minutos obtém-se um arquivo com as estruturas (mmCIF) e as métricas de confiança (pLDDT, PAE, pTM/ipTM). Em contrapartida, há um número limitado de previsões por dia, o uso está restrito a fins não comerciais e não é possível automatizar submissões nem usar ligandos arbitrários.

**ColabFold.** Desenvolvido pela comunidade académica, o ColabFold substitui as pesquisas de homólogos do AF2 (HMMER/HH-suite sobre bases de dados de vários TB) por um servidor público **MMseqs2**, tornando a construção do alinhamento múltiplo 40 a 60 vezes mais rápida sem perda relevante de exatidão \cite{mirdita2022colabfold}. Corre num *notebook* Google Colab com GPU gratuita, onde basta colar a sequência (cadeias de um complexo separadas por "`:`"), ou localmente através do pacote `colabfold`, disponível no **Bioconda** (v1.5.5) e no projeto *LocalColabFold*. É a forma mais prática de usar o AF2 sem infraestrutura própria e permite controlar parâmetros como o número de modelos, de ciclos de *recycling* ou o uso de *templates*.

**AlphaFold Protein Structure Database (AFDB).** Mantida pelo EMBL-EBI em parceria com a DeepMind, disponibiliza previsões AF2 já calculadas para mais de 214 milhões de sequências da UniProt \cite{varadi2022afdb,varadi2024afdb}, sob licença CC BY 4.0. Cada entrada (pesquisável por nome ou acesso UniProt) inclui o modelo 3D colorido por pLDDT, o mapa de PAE e ficheiros PDB/mmCIF; há ainda uma API REST pública e descarga em massa. Por exemplo, `https://alphafold.ebi.ac.uk/api/prediction/P04637` devolve as ligações para o modelo da p53 humana (versão 6 dos modelos). A AFDB não exige qualquer computação, mas limita-se a cadeias proteicas naturais isoladas.

**Instalação local.** O AF2 é instalado através de **Docker** (imagem construída a partir do repositório oficial) e requer Linux, uma GPU NVIDIA e cerca de 556 GB de bases de dados genéticas comprimidas (2,62 TB descomprimidas), existindo uma versão reduzida com menores requisitos de disco \cite{af2github}. O AF3 exige uma GPU com *compute capability* $\geq$ 8.0 (validado em A100 e H100 de 80 GB para até 5120 *tokens*), pelo menos 64 GB de RAM e cerca de 630 GB de bases de dados; a instalação é também por Docker, com instruções oficiais para conversão em **Singularity/Apptainer** para utilização em clusters \cite{af3github}. Nenhuma das versões oficiais está no Bioconda, embora todas as dependências externas (HMMER, HH-suite, Kalign, OpenMM) o estejam. Existem ainda portes para outro hardware, como a versão do AF2 otimizada pela Intel para CPUs Xeon \cite{intelopenomics}.

## Interface

O AlphaFold não tem uma interface gráfica local: as versões oficiais correm por **linha de comandos** (`run_alphafold.py`, com input FASTA no AF2 e JSON no AF3), o ColabFold funciona como *notebook* interativo ou CLI (`colabfold_batch`), e o AlphaFold Server e a AFDB são **aplicações web**. Para visualizar e analisar os resultados recorre-se a ferramentas externas como PyMOL, ChimeraX ou o visualizador Mol\* integrado nas páginas web.

# Algoritmo e Fluxo de Trabalho

## Princípio algorítmico
\afazer{AF2: MSA + templates → Evoformer (48 blocos; representações MSA e pair; atualizações e atenção triangulares) → Structure Module (Invariant Point Attention) → recycling → relaxação Amber; perda FAPE. AF3: Pairformer + módulo de difusão sobre coordenadas atómicas.}

## Fluxo de trabalho
\afazer{Figura própria do workflow (inputs → passos intermédios → outputs: PDB/mmCIF, pLDDT, PAE, pTM/ipTM).}

## Validação e interpretação dos resultados
\afazer{pLDDT (>90 muito alta; 70–90 confiante; 50–70 baixa; <50 muito baixa/desordem), PAE, pTM/ipTM; comparação com estruturas experimentais (RMSD, TM-score, lDDT \cite{mariani2013lddt}).}

# Caso de Estudo: AlphaFold 2 vs. AlphaFold 3

\afazer{Proteína escolhida; previsão AF2 (ColabFold) e AF3 (AlphaFold Server); gráficos pLDDT/PAE; comparação com estrutura experimental (PDB).}

# Reprodutibilidade, Comunidade e Limitações

## Licenciamento
\afazer{AF2: código Apache 2.0, parâmetros CC BY 4.0. AF3: código Apache 2.0 (confirmado nov/2026), pesos sob 'AlphaFold 3 Model Parameters Terms of Use' (só se recebidos diretamente da Google); uso comercial via Google Cloud. AFDB: CC BY 4.0.}

## Comunidade e manutenção
\afazer{GitHub google-deepmind/alphafold e alphafold3 (estrelas, releases, issues); AlphaFold DB com >214 milhões de estruturas \cite{varadi2024afdb}; ColabFold \cite{mirdita2022colabfold}; OpenFold \cite{ahdritz2024openfold}.}

## Limitações
\afazer{Conformação única/estática; regiões desordenadas; efeito de mutações pontuais; dependência da profundidade do MSA; requisitos de GPU/disco; restrições de licença do AF3.}

# Conclusão

\afazer{Síntese e perspetivas.}